In [1]:
pip install pyspark

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install findspark

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import findspark
findspark.init()

In [4]:
# Loading pyspark libraries for KMeans
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyspark import SparkContext
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.master("spark://192.168.0.109:7077").appName("Retail-Marketing-KMeans-2").getOrCreate()
spark

In [6]:
# Loading original dataset
df = pd.read_csv("augmented_data.csv")
df.head()

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19.0,15.0,39.0
1,2,Male,21.0,15.0,81.0
2,3,Female,20.0,16.0,6.0
3,4,Female,23.0,16.0,77.0
4,5,Female,31.0,17.0,40.0


In [ ]:
# Augmenting the data

# Replicate the data
df_augmented = pd.concat([df] * 50, ignore_index=True)

# Reset CustomerID to ensure uniqueness
df_augmented['CustomerID'] = range(1, len(df_augmented) + 1)

# Save to a CSV file
df_augmented.to_csv('Augmented_Data.csv', index=False)

In [6]:
df = spark.read.csv("augmented_data_2.csv",header=True, inferSchema=True)

In [7]:
df = df.dropna()
# Display the first few rows of the cleaned DataFrame
df.show(5)

+----------+------+----+------------------+----------------------+
|CustomerID| Genre| Age|Annual Income (k$)|Spending Score (1-100)|
+----------+------+----+------------------+----------------------+
|         1|  Male|19.0|              15.0|                  39.0|
|         2|  Male|21.0|              15.0|                  81.0|
|         3|Female|20.0|              16.0|                   6.0|
|         4|Female|23.0|              16.0|                  77.0|
|         5|Female|31.0|              17.0|                  40.0|
+----------+------+----+------------------+----------------------+
only showing top 5 rows



In [8]:
df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Annual Income (k$): double (nullable = true)
 |-- Spending Score (1-100): double (nullable = true)



In [ ]:
#!pip show pyspark

In [9]:
from pyspark.sql import functions as F

# Encoding Gender
df = df.withColumn("Gender", F.when(df["Genre"] == "Male", 0).otherwise(1))

# Selecting desired columns
df = df.select("Gender", "Age", "Annual Income (k$)", "Spending Score (1-100)")

In [10]:
from pyspark.ml.feature import VectorAssembler
features = tuple(df.columns)
features

('Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)')

In [11]:
# Making a spark dataframe
assembler = VectorAssembler(inputCols=features,outputCol="features")
dataset = assembler.transform(df)
dataset.select("features").show(truncate=False)

+--------------------+
|features            |
+--------------------+
|[0.0,19.0,15.0,39.0]|
|[0.0,21.0,15.0,81.0]|
|[1.0,20.0,16.0,6.0] |
|[1.0,23.0,16.0,77.0]|
|[1.0,31.0,17.0,40.0]|
|[1.0,22.0,17.0,76.0]|
|[1.0,35.0,18.0,6.0] |
|[1.0,23.0,18.0,94.0]|
|[0.0,64.0,19.0,3.0] |
|[1.0,30.0,19.0,72.0]|
|[0.0,67.0,19.0,14.0]|
|[1.0,35.0,19.0,99.0]|
|[1.0,58.0,20.0,15.0]|
|[1.0,24.0,20.0,77.0]|
|[0.0,37.0,20.0,13.0]|
|[0.0,22.0,20.0,79.0]|
|[1.0,35.0,21.0,35.0]|
|[0.0,20.0,21.0,66.0]|
|[0.0,52.0,23.0,29.0]|
|[1.0,35.0,23.0,98.0]|
+--------------------+
only showing top 20 rows



In [12]:
# Creating KMeans #52mins
kmeans = KMeans().setK(5).setSeed(42)
model = kmeans.fit(dataset)
predictions = model.transform(dataset)

In [14]:
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)
print("Silhouetter with squared euclidean distance = {}".format(silhouette))

Silhouetter with squared euclidean distance = 0.6005973206530004


In [15]:
print("Cluster Centers: ")
ctr = []
centers = model.clusterCenters()

for center in centers:
    ctr.append(center)
    print(center)

Cluster Centers: 
[ 0.6  24.8  41.46 63.7 ]
[ 0.53846065 32.69231806 86.5384327  82.12821972]
[ 0.59090909 44.31818182 25.77272727 20.27272727]
[ 0.48648649 40.32432432 87.43243243 18.18918919]
[ 0.57692379 53.61539771 55.03844185 48.71155537]


In [16]:
centers = pd.DataFrame(ctr,columns=features)
centers.head()

,Gender,Age,Annual Income (k$),Spending Score (1-100)
0,0.600000,24.800000,41.460000,63.700000
1,0.538461,32.692318,86.538433,82.128220
2,0.590909,44.318182,25.772727,20.272727
3,0.486486,40.324324,87.432432,18.189189
4,0.576924,53.615398,55.038442,48.711555
